In [ ]:
# ===================== 仅保留库导入、全局参数、工具函数（导入时不会执行任何输出） =====================
# 基础数值、表格库
import numpy as np
import pandas as pd
# 绘图
import matplotlib.pyplot as plt
import seaborn as sns
# 预处理
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
# 深度学习tensorflow/keras
import tensorflow as tf
from tensorflow.keras.models import Sequential, load_model
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
# 解决jupyter中文、负号显示问题
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
sns.set_style("whitegrid")

# 固定随机种子，保证每次训练结果一致
np.random.seed(42)
tf.random.set_seed(42)

# 全局配置参数（无需修改）
file_path = r"C:\Users\dengj\Desktop\基于 LSTM 的城市空气质量指数（AQI）预测\Beijing Multisite air Quality data.csv"
target_station = "Aotizhongxin"
look_back = 24
epochs = 100
batch_size = 64
# 固定数值特征列表
numeric_fea = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'WSPM', 'hour', 'month', 'season']
cate_fea = ['wd']

# ========== 工具函数（仅定义，导入不运行） ==========
def create_window_data(dataset, lookback, target_idx=0):
    X, Y = [], []
    for i in range(lookback, len(dataset)):
        X.append(dataset[i-lookback:i, :])
        Y.append(dataset[i, target_idx])
    return np.array(X), np.array(Y)

def inverse_pm25_simple(scaled_target, scaler):
    return scaler.inverse_transform(scaled_target)[:,0]

def get_aqi_level(pm):
    if pm <= 35:
        return "优"
    elif pm <=75:
        return "良"
    elif pm <=115:
        return "轻度污染"
    elif pm <=150:
        return "中度污染"
    elif pm <=250:
        return "重度污染"
    else:
        return "严重污染"

# ========== 模型结构定义（仅定义网络，不训练） ==========
def build_lstm_model(input_dim):
    model = Sequential()
    model.add(LSTM(128, return_sequences=True, input_shape=(look_back, input_dim)))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(LSTM(64, return_sequences=False))
    model.add(BatchNormalization())
    model.add(Dropout(0.3))
    model.add(Dense(32, activation="relu"))
    model.add(Dense(1))
    opt = Adam(learning_rate=0.001)
    model.compile(optimizer=opt, loss="mse", metrics=["mae"])
    return model

# ===================== 核心执行函数：所有打印、绘图、训练全部放这里 =====================
# 导入LSTM文件时不会自动运行，只有手动调用才会执行并输出结果
def run_experiment():
    # 1.读取数据、筛选站点、时间处理
    df = pd.read_csv(file_path)
    df = df[df['station'] == target_station].reset_index(drop=True)
    df['time'] = pd.to_datetime(df[['year', 'month', 'day', 'hour']])
    df = df.set_index('time').sort_index()

    # 打印数据集信息
    print("====数据集基础信息====")
    print(f"监测站点：{target_station}")
    print(f"数据起始：{df.index.min()}")
    print(f"数据结束：{df.index.max()}")
    print(f"总样本条数：{len(df)}")
    print("\n前5行数据预览：")
    print(df.head())
    print("\n各列缺失值统计：")
    print(df.isnull().sum())

    # PM2.5时序图
    plt.figure(figsize=(16, 5))
    plt.plot(df.index, df['PM2.5'], color='#1f77b4', linewidth=1.2, label='PM2.5 浓度μg/m³')
    plt.title(f'{target_station}站点PM2.5时序变化趋势', fontsize=14)
    plt.xlabel("时间")
    plt.ylabel("PM2.5 浓度")
    plt.legend()
    plt.tight_layout()
    plt.show()

    # 相关性热力图
    feature_cols = ['PM2.5', 'PM10', 'SO2', 'NO2', 'CO', 'O3', 'TEMP', 'PRES', 'DEWP', 'WSPM']
    corr_df = df[feature_cols].corr()
    plt.figure(figsize=(12, 8))
    sns.heatmap(corr_df, annot=True, cmap="Blues", vmin=-1, vmax=1, fmt=".2f")
    plt.title("污染物&气象特征相关性热力图", fontsize=14)
    plt.tight_layout()
    plt.show()

    # 缺失值、异常值清洗
    df = df.ffill()
    df = df.bfill()
    print("填充后总缺失值数量：", df.isnull().sum().sum())
    df['PM2.5'] = pd.to_numeric(df['PM2.5'], errors='coerce')
    df['PM10'] = pd.to_numeric(df['PM10'], errors='coerce')
    df['PM2.5'] = df['PM2.5'].clip(lower=0, upper=500)
    df['PM10'] = df['PM10'].clip(lower=0, upper=600)
    df['PM2.5'].fillna(df['PM2.5'].mean(), inplace=True)
    df['PM10'].fillna(df['PM10'].mean(), inplace=True)

    # 构造时间特征
    df['hour'] = df.index.hour
    df['month'] = df.index.month
    df['season'] = df.index.month.map({
        1:1,2:1,3:2,4:2,5:2,
        6:3,7:3,8:3,9:4,10:4,11:4,12:1
    })
    print("数据清洗与时间特征构造完成！")
    print(df[['hour','month','season']].head())

    # 特征归一化、编码
    pm25_scaler = MinMaxScaler()
    pm25_scaler.fit(df[['PM2.5']])
    scaler_num = MinMaxScaler(feature_range=(0,1))
    df_numeric = scaler_num.fit_transform(df[numeric_fea])
    encoder_wd = OneHotEncoder(sparse_output=False, drop="first")
    df_cate = encoder_wd.fit_transform(df[cate_fea])
    data_processed = np.hstack([df_numeric, df_cate])
    n_features = data_processed.shape[1]
    print(f"预处理后总特征维度：{n_features}")
    print("处理后数据形状：", data_processed.shape)

    # 生成时序样本
    X_all, Y_all = create_window_data(data_processed, look_back)
    print(f"全部样本总数：{len(X_all)}")
    print(f"输入形状 [样本, 时间步, 特征]：{X_all.shape}")
    print(f"输出形状 [样本]：{Y_all.shape}")

    # 数据集划分
    total_num = len(X_all)
    train_split = int(total_num * 0.7)
    val_split = int(total_num * 0.85)
    X_train = X_all[:train_split]
    Y_train = Y_all[:train_split]
    X_val = X_all[train_split:val_split]
    Y_val = Y_all[train_split:val_split]
    X_test = X_all[val_split:]
    Y_test = Y_all[val_split:]
    print(f"训练集：{X_train.shape}")
    print(f"验证集：{X_val.shape}")
    print(f"测试集：{X_test.shape}")

    # 搭建模型
    model = build_lstm_model(n_features)
    model.summary()

    # 回调函数
    early_stop = EarlyStopping(monitor="val_loss", patience=10, restore_best_weights=True, verbose=1)
    lr_down = ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=5, min_lr=1e-6, verbose=1)

    # 训练
    print("=====开始训练=====")
    history = model.fit(
        X_train, Y_train,
        batch_size=batch_size,
        epochs=epochs,
        validation_data=(X_val, Y_val),
        callbacks=[early_stop, lr_down],
        verbose=1
    )

    # 保存模型
    save_path = r"C:\Users\dengj\Desktop\基于 LSTM 的城市空气质量指数（AQI）预测\lstm_pm25_model.h5"
    model.save(save_path)
    print(f"模型已保存至：{save_path}")

    # 损失曲线绘图
    plt.figure(figsize=(12, 5))
    plt.plot(history.history["loss"], label="训练损失MSE", color="#1f77b4")
    plt.plot(history.history["val_loss"], label="验证损失MSE", color="#ff7f0e")
    plt.title("模型训练与验证损失变化曲线")
    plt.xlabel("Epoch迭代轮次")
    plt.ylabel("MSE损失值")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # 预测、逆归一化
    y_pred_scaled = model.predict(X_test, verbose=0)
    y_pred_real = inverse_pm25_simple(y_pred_scaled, pm25_scaler)
    y_true_real = inverse_pm25_simple(Y_test.reshape(-1,1), pm25_scaler)

    # 计算指标
    rmse = np.sqrt(mean_squared_error(y_true_real, y_pred_real))
    mae = mean_absolute_error(y_true_real, y_pred_real)
    r2 = r2_score(y_true_real, y_pred_real)
    mape = np.mean(np.abs((y_true_real - y_pred_real) / (y_true_real + 1e-7))) * 100
    print("========模型评估指标========")
    print(f"RMSE 均方根误差：{rmse:.2f}")
    print(f"MAE 平均绝对误差：{mae:.2f}")
    print(f"R² 决定系数：{r2:.4f}")
    print(f"MAPE 平均百分比误差：{mape:.2f}%")

    # 测试时间轴
    test_start_idx = look_back + train_split + len(X_val)
    test_time = df.index[test_start_idx : test_start_idx + len(y_true_real)]

    # 真实vs预测曲线
    plt.figure(figsize=(16,6))
    plt.plot(test_time, y_true_real, color="#1f77b4", label="真实PM2.5")
    plt.plot(test_time, y_pred_real, color="red", alpha=0.7, label="LSTM预测PM2.5")
    plt.title("PM2.5真实值与预测值对比曲线")
    plt.xlabel("时间")
    plt.ylabel("PM2.5 浓度 μg/m³")
    plt.legend()
    plt.grid(alpha=0.3)
    plt.show()

    # 残差图
    residual = y_true_real - y_pred_real
    plt.figure(figsize=(10,4))
    sns.histplot(residual, bins=40, kde=True, color="#2ca02c")
    plt.axvline(x=0, color="red", linestyle="--")
    plt.title("预测残差分布直方图")
    plt.xlabel("误差(真实-预测)")
    plt.ylabel("样本数量")
    plt.show()

    # 空气质量等级统计
    true_levels = [get_aqi_level(x) for x in y_true_real]
    pred_levels = [get_aqi_level(x) for x in y_pred_real]
    stat_df = pd.DataFrame({
        "真实等级数量": pd.Series(true_levels).value_counts(),
        "预测等级数量": pd.Series(pred_levels).value_counts()
    })
    print("空气质量等级统计：")
    print(stat_df)
    stat_df.plot(kind="bar", figsize=(10,5))
    plt.title("真实/预测空气质量等级对比")
    plt.ylabel("样本数量")
    plt.grid(axis="y", alpha=0.3)
    plt.show()

    # 将所有结果打包返回，供主文件调用
    result_dict = {
        "df": df,
        "X_train": X_train, "Y_train": Y_train,
        "X_val": X_val, "Y_val": Y_val,
        "X_test": X_test, "Y_test": Y_test,
        "model": model,
        "history": history,
        "y_true_real": y_true_real,
        "y_pred_real": y_pred_real,
        "test_time": test_time,
        "residual": residual,
        "stat_df": stat_df,
        "rmse": rmse,
        "mae": mae,
        "r2": r2,
        "mape": mape,
        "pm25_scaler": pm25_scaler,
        "corr_df": corr_df
    }
    return result_dict